# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a step-by-step workflow for loading, exploring, and analyzing a clinical oncology dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema accessible via the following URL:
- https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json


In [ ]:
# Ensure `mlcroissant` is installed (uncomment if running for the first time)
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and initialize a `mlcroissant.Dataset` for exploration. All objects will be accessed and referenced using their `@id` field.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant dataset
ds = mlc.Dataset(croissant_url)

# Access dataset metadata
dataset_meta = ds.metadata
print(dataset_meta.name)
print(dataset_meta.description)
print(f"Version: {dataset_meta.version}")
print(f"Published: {dataset_meta.datePublished}")
print(f"Number of record sets: {len(dataset_meta.recordSet)}")

## 2. Data Overview
Review the available record sets and their field schema. Each record set and its fields are identified using their `@id`.

In [ ]:
# Show details for each record set and its fields by @id.
if not dataset_meta.recordSet:
    print("This dataset schema does not enumerate recordSet in the top-level metadata.")
    # mlcroissant will still parse available record sets internally
    available_record_sets = [rs['@id'] for rs in ds._croissant['recordSet']] if 'recordSet' in ds._croissant else []
else:
    available_record_sets = [getattr(rs, '@id', None) for rs in dataset_meta.recordSet]
print("Available record sets (by @id):")
for i, rec_id in enumerate(available_record_sets):
    print(f"{i+1}. {rec_id}")

# For each record set, print field @id information
for rec_id in available_record_sets:
    print(f"\nRecord set: {rec_id}")
    # Retrieve the record set schema
    rec_schema = next((rs for rs in ds._croissant['recordSet'] if rs['@id'] == rec_id), None)
    if rec_schema and 'field' in rec_schema:
        print("  Fields:")
        for fld in rec_schema['field']:
            fid = fld['@id']
            ftype = fld.get('dataType')
            print(f"    - {fid} (type: {ftype})")
    else:
        print("  [No fields listed]")

## 3. Data Extraction
Load records for each record set by its `@id`, producing Pandas DataFrames for further analysis. All references use the entity's `@id`.

In [ ]:
# Load records for each record set into DataFrames (by @id)
dataframes = {}

for rec_id in available_record_sets:
    print(f"Loading data for {rec_id} ...")
    records_iter = ds.records(record_set=rec_id)
    df = pd.DataFrame(list(records_iter))
    dataframes[rec_id] = df
    print(f" - {df.shape[0]} records, columns: {df.columns.tolist()}")
    if not df.empty:
        display(df.head(3))

# Choose one main record set for detailed analysis
# We'll use the first available record set
main_record_set_id = available_record_sets[0] if available_record_sets else None
if main_record_set_id:
    print(f"\nMain record set selected for analysis: {main_record_set_id}")
    print(f"Columns: {dataframes[main_record_set_id].columns.tolist()}")
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply basic data selection, filtering, and normalization using only `@id`-referenced columns. Here we select a numeric field for analysis, filter rows, normalize numeric fields, and group by a categorical attribute.

In [ ]:
# Pick a numeric field by @id for demonstration
main_df = dataframes[main_record_set_id]

# List numeric-like columns to help selection
numeric_candidates = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])]
print(f"Numeric fields in main record set: {numeric_candidates}")

# For this dataset, demonstration: choose the first numeric column
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    print(f"Using numeric field: {numeric_field_id}")

    # Choose a filtering threshold (median as example)
    threshold = main_df[numeric_field_id].median()
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (median):")
    display(filtered_df.head())

    # Normalize the numeric column (Z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Identify a group field (categorical), excluding the numeric field
    group_candidates = [col for col in main_df.columns if col != numeric_field_id and main_df[col].dtype == 'object']
    if group_candidates:
        group_field_id = group_candidates[0]
        print(f"Grouping by field: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable group/categorical field found for grouping.")
else:
    print("No numeric fields detected in the main DataFrame.")

## 5. Visualization
Display visualizations to help understand the distribution of numeric and categorical fields referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of the main numeric field (if available)
if main_record_set_id and numeric_candidates:
    plt.figure(figsize=(7, 4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=15, kde=True, color='royalblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field exists, show a boxplot
    if group_candidates:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we've loaded and explored the clinical dataset using `mlcroissant` referencing all schema entities by their `@id`. We retrieved metadata, inspected available record sets and fields, loaded records into DataFrames, filtered and normalized data (using only columns by `@id`), grouped by a selected attribute, and visualized data distributions. You can extend these analyses based on your use case, always referencing desired fields and record sets with their `@id` for consistent and reproducible data access.
